# Model Comparison: Claude Sonnet 4.6 (Teacher) vs Student

This notebook compares the performance of two models on the SQL generation task using **Anthropic API**:

- **Teacher Model**: Claude Sonnet 4.6 (via Anthropic API)
- **Student Model**: Llama 3.1 8B fine-tuned with QLoRA

## Requirements

- Anthropic API key set in `.env` (ANTHROPIC_API_KEY)
- Configured `config/base.yaml` with `provider: anthropic` and `model: claude-sonnet-4-6`

## Objectives

1. Validate the distillation approach by measuring quality parity
2. Compare inference latency and throughput
3. Analyze cost-effectiveness at different query volumes
4. Provide recommendations for deployment scenarios

## Hypothesis

The student model should:
- Approach teacher quality (>75% score ratio)
- Offer significantly lower cost at scale
- Provide faster inference for local deployment

In [ ]:
# Cell 1: Imports, NLTK setup, and path configuration
import json
import jsonlines
import logging
import time
from pathlib import Path
from typing import Any, Dict, List
import re

import nltk
import torch
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# Download NLTK data to prevent import hang
print("Downloading NLTK data...")
nltk.download('punkt')  # Remove quiet=True to see progress
nltk.download('punkt_tab')  # Remove quiet=True to see progress
print("✅ NLTK data ready")

# Project imports
import sys
sys.path.append("..")

print("Importing evaluation modules...")
from src.evaluate.benchmark import BenchmarkRunner, _build_prompt
print("  ✅ benchmark")

from src.evaluate.judge import LLMJudge, JudgeExample
print("  ✅ judge")

from src.evaluate.metrics import evaluate_batch
print("  ✅ metrics")

from src.llm.client import TeacherClient, Message
print("  ✅ llm.client")

# Configuration
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Initialize model loading flag
model_loaded = False

def find_project_root() -> Path:
    """Find project root by searching for marker files."""
    MARKER_FILES = ["CLAUDE.md", "pyproject.toml", ".git"]
    current_path = Path.cwd()
    
    for parent in [current_path] + list(current_path.parents):
        for marker in MARKER_FILES:
            marker_path = parent / marker
            if marker_path.exists():
                logger.info(f"Found project root at: {parent} (marker: {marker})")
                return parent
    
    if "notebooks" in current_path.parts:
        notebooks_index = current_path.parts.index("notebooks")
        potential_root = Path(*current_path.parts[:notebooks_index])
        if (potential_root / "CLAUDE.md").exists():
            logger.info(f"Found project root via notebooks/ path: {potential_root}")
            return potential_root
    
    raise RuntimeError(f"Cannot find project root from {current_path}")

# Get project root and setup absolute paths
print("Finding project root...")
PROJECT_ROOT = find_project_root()
print(f"  ✅ Project root: {PROJECT_ROOT}")

# Constants - use absolute paths
BASE_MODEL_ID = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"
LORA_PATH = PROJECT_ROOT / "models" / "sql-llama-8b-lora"
TEST_DATA_PATH = PROJECT_ROOT / "data" / "curated" / "test.jsonl"
CONFIG_PATH = PROJECT_ROOT / "config" / "base.yaml"

# Cost constants (USD per 1M tokens) - Sonnet pricing
SONNET_INPUT_COST = 3.00
SONNET_OUTPUT_COST = 15.00
TRAINING_COST = 0.50  # One-time training cost (local compute)

# Verify paths exist
print("Verifying paths...")
if not TEST_DATA_PATH.exists():
    raise FileNotFoundError(f"Test data not found at: {TEST_DATA_PATH}")
if not LORA_PATH.exists():
    raise FileNotFoundError(f"LoRA adapter not found at: {LORA_PATH}")
print(f"  ✅ Test data: {TEST_DATA_PATH}")
print(f"  ✅ LoRA adapter: {LORA_PATH}")
print(f"  ✅ Config: {CONFIG_PATH}")

print(f"Project root: {PROJECT_ROOT}")
print(f"Test data: {TEST_DATA_PATH} (exists: {TEST_DATA_PATH.exists()})")
print(f"LoRA adapter: {LORA_PATH} (exists: {LORA_PATH.exists()})")

/mnt/d/GitHub/model-tailor/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ NLTK data ready
Importing evaluation modules...


[nltk_data] Downloading package punkt to /home/jamestjy/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/jamestjy/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
2026-04-20 20:26:15,639 - INFO - Found project root at: /mnt/d/GitHub/model-tailor (marker: CLAUDE.md)


  ✅ benchmark
  ✅ judge
  ✅ metrics
  ✅ llm.client
Finding project root...
  ✅ Project root: /mnt/d/GitHub/model-tailor
Verifying paths...
  ✅ Test data: /mnt/d/GitHub/model-tailor/data/curated/test.jsonl
  ✅ LoRA adapter: /mnt/d/GitHub/model-tailor/models/sql-llama-8b-lora
  ✅ Config: /mnt/d/GitHub/model-tailor/config/base.yaml
Project root: /mnt/d/GitHub/model-tailor
Test data: /mnt/d/GitHub/model-tailor/data/curated/test.jsonl (exists: True)
LoRA adapter: /mnt/d/GitHub/model-tailor/models/sql-llama-8b-lora (exists: True)


## 1. Load Test Data

In [2]:
# Load test dataset
test_data = []
with jsonlines.open(TEST_DATA_PATH) as f:
    test_data = list(f)

print(f"Loaded {len(test_data)} test examples")
print(f"\nSample example:")
print(f"  NL:  {test_data[0]['natural_language']}")
print(f"  SQL: {test_data[0]['sql']}")
print(f"  Difficulty: {test_data[0]['difficulty']}")
print(f"  Category: {test_data[0]['category']}")

# Analyze distribution
df = pd.DataFrame(test_data)
print(f"\nDataset distribution:")
print(f"  By difficulty: {df['difficulty'].value_counts().to_dict()}")
print(f"  By category: {df['category'].value_counts().to_dict()}")

Loaded 14 test examples

Sample example:
  NL:  What is the average salary of all employees?
  SQL: SELECT AVG(salary) AS average_salary FROM employees;
  Difficulty: easy
  Category: aggregation

Dataset distribution:
  By difficulty: {'medium': 6, 'easy': 4, 'hard': 3, 'expert': 1}
  By category: {'select': 5, 'aggregation': 2, 'subquery with aggregation': 1, 'Subquery + NOT IN': 1, 'DML': 1, 'update': 1, 'CTE + Window Function (LAG)': 1, 'DML with CTE and LIMIT/OFFSET': 1, 'aggregate': 1}


## 2. Load Models

In [3]:
# Skip loading student model due to RAM constraints
# Llama 3.1 8B requires ~16GB RAM for CPU inference
# Your system has only ~1.6GB available

print("⚠️  Skipping student model loading - insufficient RAM")
print(f"System requires ~16GB, only ~1.6GB available")

model_loaded = False
student_model = None
tokenizer = None

# Initialize teacher client with absolute config path
teacher_client = TeacherClient(config_path=str(CONFIG_PATH))
print(f"\nTeacher client initialized: {teacher_client.provider} / {teacher_client.model}")

print("\n⚠️  Only teacher evaluation will be performed.")
print("To evaluate student model, use a machine with 16GB+ RAM")

⚠️  Skipping student model loading - insufficient RAM
System requires ~16GB, only ~1.6GB available

Teacher client initialized: anthropic / claude-sonnet-4-6

⚠️  Only teacher evaluation will be performed.
To evaluate student model, use a machine with 16GB+ RAM


## 3. Evaluate Teacher Model (Sonnet)

## Teacher Model Evaluation

This cell evaluates Claude Sonnet 4.6 on the test set:
- Sends each natural language query to the API
- Extracts SQL from the response
- Tracks time and cost

**Estimated cost:** ~$0.03 (3 cents)
**Estimated time:** ~60 seconds

In [5]:
def evaluate_teacher(test_data: List[Dict], client: TeacherClient) -> tuple:
    """Evaluate Sonnet on test set.
    
    Returns:
        (predictions, elapsed_time, cost_breakdown)
    """
    predictions = []
    total_input_tokens = 0
    total_output_tokens = 0
    
    t0 = time.perf_counter()
    
    print(f"Evaluating teacher model on {len(test_data)} examples...")
    
    for i, example in enumerate(test_data):
        prompt = _build_prompt(example["natural_language"])
        messages = [Message(role="user", content=prompt)]
        
        try:
            response = client.complete(messages, temperature=0.0, max_tokens=256)
            
            # Extract SQL - remove markdown code blocks if present
            sql = response.strip()
            if sql.startswith("```"):
                # Remove ```sql and ``` markers
                lines = sql.split("\n")
                sql = "\n".join(lines[1:-1]) if len(lines) > 2 else sql
            sql = sql.strip().rstrip(";")
            predictions.append(sql)
            
            # Estimate tokens (rough approximation: 1 token ≈ 4 chars)
            total_input_tokens += len(prompt) // 4
            total_output_tokens += len(response) // 4
            
            if (i + 1) % 5 == 0:
                print(f"  Progress: {i + 1}/{len(test_data)} examples")
                
        except Exception as e:
            logger.error(f"Error on example {i}: {e}")
            predictions.append("")
    
    elapsed = time.perf_counter() - t0
    
    # Calculate API cost
    input_cost = (total_input_tokens / 1_000_000) * SONNET_INPUT_COST
    output_cost = (total_output_tokens / 1_000_000) * SONNET_OUTPUT_COST
    total_cost = input_cost + output_cost
    
    cost_breakdown = {
        "input_tokens": total_input_tokens,
        "output_tokens": total_output_tokens,
        "input_cost_usd": round(input_cost, 4),
        "output_cost_usd": round(output_cost, 4),
        "total_cost_usd": round(total_cost, 4),
    }
    
    print(f"\nTeacher evaluation complete: {elapsed:.1f}s, ${total_cost:.4f}")
    return predictions, elapsed, cost_breakdown

# Run teacher evaluation
teacher_predictions, teacher_time, teacher_cost = evaluate_teacher(test_data, teacher_client)

print(f"\nTeacher results:")
print(f"  Time: {teacher_time:.2f}s")
print(f"  Throughput: {len(test_data)/teacher_time:.3f} examples/sec")
print(f"  Cost: ${teacher_cost['total_cost_usd']:.4f}")
print(f"  Avg latency: {teacher_time/len(test_data):.3f}s per query")

Evaluating teacher model on 14 examples...


2026-04-20 20:28:53,152 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-04-20 20:28:57,850 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-04-20 20:29:01,764 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-04-20 20:29:05,673 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-04-20 20:29:10,149 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"


  Progress: 5/14 examples


2026-04-20 20:29:13,741 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-04-20 20:29:16,755 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-04-20 20:29:20,334 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-04-20 20:29:24,543 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-04-20 20:29:33,461 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"


  Progress: 10/14 examples


2026-04-20 20:29:37,759 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-04-20 20:29:42,874 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-04-20 20:29:48,045 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2026-04-20 20:29:49,413 - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"



Teacher evaluation complete: 59.4s, $0.0330

Teacher results:
  Time: 59.44s
  Throughput: 0.236 examples/sec
  Cost: $0.0330
  Avg latency: 4.246s per query


## Clean Predictions

The model responses sometimes include explanation text after the SQL.
This cell extracts only the SQL queries by:
- Finding ```sql code blocks
- Removing the markdown markers
- Stopping at the closing ```
- Removing trailing semicolons

This ensures clean SQL for evaluation.

In [ ]:
# Aggressive cleanup - remove everything after ```import reteacher_predictions_clean = []for pred in teacher_predictions:    # Remove everything after ``` or ```sql    if '```' in pred:        pred = pred.split('```')[0].strip()        # Remove trailing semicolon    pred = pred.rstrip(';').strip()    teacher_predictions_clean.append(pred)print(f"Cleaned {len(teacher_predictions_clean)} predictions")print("\nFirst 3 cleaned predictions:")for i in range(3):    print(f"{i+1}. {teacher_predictions_clean[i]}")

## Save Results

Save the cleaned teacher predictions to disk for later comparison.

Output: `tasks/sql_generation/teacher_results_anthropic.json`

This file can be loaded by the comparison notebook once student
model results are available.

In [10]:
results = {
    "metadata": {
        "test_examples": len(test_data),
        "teacher_model": "claude-sonnet-4-6",
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S")
    },
    "teacher": {
        "predictions": teacher_predictions_clean,
        "elapsed_seconds": round(teacher_time, 2),
        "cost_breakdown": teacher_cost
    }
}

output_path = Path("tasks/sql_generation/teacher_results_anthropic.json")
output_path.parent.mkdir(parents=True, exist_ok=True)

with open(output_path, "w") as f:
    json.dump(results, f, indent=2)

print(f"\n✅ Teacher results saved to: {output_path}")


✅ Teacher results saved to: tasks/sql_generation/teacher_results_anthropic.json


In [ ]:
for i in range(5):
    print(f"{i+1}. NL: {test_data[i]['natural_language']}")
    print(f"   Pred: {teacher_predictions_clean[i]}")
    print(f"   Target: {test_data[i]['sql']}")
    print()